# Plots for the ICML paper

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgba

from rewarduq.utils_ext.plot import Plotter
from rewarduq.utils_ext.tools import setup_logging

setup_logging()
plt.ioff()
plt.set_loglevel("WARNING")

logger = logging.getLogger(__name__)

PATH_DATA = Path("data")
PATH_OUTPUT = Path("../../output/plots/icml_paper")

FONTSIZE_TINY = 6
FONTSIZE_SMALL = 8
FONTSIZE_DEFAULT = 9
FONTSIZE_LARGE = 10

# Setup plotter
Plotter.setup(css_patches=["overflow_auto", "gray_background"])
Plotter.configure(
    basewidth=5.5,
    fontsize={
        "default": FONTSIZE_DEFAULT,
        "legend": FONTSIZE_SMALL,
        "label": FONTSIZE_SMALL,
        "xtick": FONTSIZE_SMALL,
        "ytick": FONTSIZE_SMALL,
    },
    latex=True,
    latex_preamble="\n".join(
        [
            r"\usepackage[utf8]{inputenc}",
            r"\usepackage[T1]{fontenc}",
            r"\usepackage{microtype}",
            r"\usepackage{amsmath,amssymb,amsfonts,mathrsfs}",
        ]
    ),
    rcparams={
        "lines.linewidth": 1,  # default: 1.5
        "axes.labelpad": 2,  # default: 4
    },
    save_dir=PATH_OUTPUT,
    save_format="pdf",
)

In [ ]:
DATASET_LABEL_MAP = {
    "trl-lib/ultrafeedback_binarized": "UltraFeedback",
    "Skywork/Skywork-Reward-Preference-80K-v0.2": "Skywork 80K v0.2",
    "allenai/llama-3.1-tulu-3-8b-preference-mixture": "Tulu 3 8B",
}

DATASET_FILENAME_MAP = {
    "trl-lib/ultrafeedback_binarized": "ultrafeedback",
    "Skywork/Skywork-Reward-Preference-80K-v0.2": "skywork",
    "allenai/llama-3.1-tulu-3-8b-preference-mixture": "tulu",
}

METHOD_SELECTOR_MAP = {
    "ens_lin": lambda df: (
        (df["pipeline"] == "mlp_head_ensemble.MLPHeadEnsemblePipeline") & (df["model.head_num_layers"] == 1)
    ),
    "ens_mlp": lambda df: (
        (df["pipeline"] == "mlp_head_ensemble.MLPHeadEnsemblePipeline") & (df["model.head_num_layers"] != 1)
    ),
    "ens_lora": lambda df: df["pipeline"] == "lora_ensemble.LoraEnsemblePipeline",
    "mcd_dpo": lambda df: df["pipeline"] == "dpo_head_dropout_ensemble.DPOHeadDropoutEnsemblePipeline",
    "bay_lin": lambda df: (
        (df["pipeline"] == "bayesian_linear_head.BayesianLinearHeadPipeline")
        & (df["trainer.final_hessian_mode"] == "unweighted")
    ),
    "bay_lin_weighted": lambda df: (
        (df["pipeline"] == "bayesian_linear_head.BayesianLinearHeadPipeline")
        & (df["trainer.final_hessian_mode"] == "weighted")
    ),
}

METHOD_LABEL_MAP = {
    "ens_lin": "ENS-LIN",
    "ens_mlp": "ENS-MLP",
    "ens_lora": "ENS-LoRA",
    "mcd_dpo": "MCD-DPO",
    "bay_lin": "BAY-LIN",
    "bay_lin_weighted": "BAY-LIN (weighted)",
}

METHOD_COLOR_MAP = {
    "ens_lin": "C5",
    "ens_mlp": "C0",
    "ens_lora": "C3",
    "mcd_dpo": "C1",
    "bay_lin": "C2",
    "bay_lin_weighted": "C4",
}

BASE_MODEL_FAMILY_MAP = {
    "Qwen3": [
        "Qwen/Qwen3-0.6B",
        "Qwen/Qwen3-1.7B",
        "Qwen/Qwen3-4B",
        "Qwen/Qwen3-8B",
        "Qwen/Qwen3-14B",
        "Qwen/Qwen3-32B",
    ],
    "Skywork-Qwen3": [
        "Skywork/Skywork-Reward-V2-Qwen3-0.6B",
        "Skywork/Skywork-Reward-V2-Qwen3-1.7B",
        "Skywork/Skywork-Reward-V2-Qwen3-4B",
        "Skywork/Skywork-Reward-V2-Qwen3-8B",
    ],
}

BASE_MODEL_SIZE_MAP = {
    "Qwen/Qwen3-0.6B": "0.6",
    "Qwen/Qwen3-1.7B": "1.7",
    "Qwen/Qwen3-4B": "4",
    "Qwen/Qwen3-8B": "8",
    "Qwen/Qwen3-14B": "14",
    "Qwen/Qwen3-32B": "32",
    "Skywork/Skywork-Reward-V2-Qwen3-0.6B": "0.6",
    "Skywork/Skywork-Reward-V2-Qwen3-1.7B": "1.7",
    "Skywork/Skywork-Reward-V2-Qwen3-4B": "4",
    "Skywork/Skywork-Reward-V2-Qwen3-8B": "8",
}

BASE_MODEL_FILENAME_MAP = {
    "Qwen/Qwen3-0.6B": "qwen3_0_6b",
    "Qwen/Qwen3-1.7B": "qwen3_1_7b",
    "Qwen/Qwen3-4B": "qwen3_4b",
    "Qwen/Qwen3-8B": "qwen3_8b",
    "Qwen/Qwen3-14B": "qwen3_14b",
    "Qwen/Qwen3-32B": "qwen3_32b",
    "Skywork/Skywork-Reward-V2-Qwen3-0.6B": "skywork_qwen3_0_6b",
    "Skywork/Skywork-Reward-V2-Qwen3-1.7B": "skywork_qwen3_1_7b",
    "Skywork/Skywork-Reward-V2-Qwen3-4B": "skywork_qwen3_4b",
    "Skywork/Skywork-Reward-V2-Qwen3-8B": "skywork_qwen3_8b",
}

METRICS_LABEL_MAP = {
    "ranking/0.0": r"$\mathrm{RS}_{0}$",
    "ranking/0.01": r"$\mathrm{RS}_{0.01}$",
    "ranking/0.2": r"$\mathrm{RS}_{0.2}$",
    "ranking/1.0": r"$\mathrm{RS}_{1}$",
    "win_rate": r"$\mathrm{win\ rate}$",
    "prefs/confident_true_rate": r"$\mathrm{CT\ rate}$",
    "prefs/confident_false_rate": r"$\mathrm{CF\ rate}$",
    "prefs/ece": r"$\mathrm{ECE}$",
    "prefs/elce": r"$\mathrm{EBCE}$",
    "prefs/euce": r"$\mathrm{EBCE}$",
}

METRICS_FILENAME_MAP = {
    "ranking/0.0": "ranking_0_0",
    "ranking/0.01": "ranking_0_01",
    "ranking/0.2": "ranking_0_2",
    "ranking/1.0": "ranking_1_0",
    "win_rate": "win_rate",
    "prefs/confident_true_rate": "ct_rate",
    "prefs/confident_false_rate": "cf_rate",
    "prefs/ece": "ece",
    "prefs/elce": "ebce",
    "prefs/euce": "ebce",
}

EVAL_CONFIG_KEYS = [
    "dataset.train.path",
    "dataset.eval.path",
    "pipeline",
    "model.base_model_name_or_path",
    # common
    "trainer.learning_rate",
    # ENS-MLP
    "model.head_num_layers",
    "trainer.center_rewards_coefficient",
    "trainer.regularization_towards_initial_weights",
    # MCD-DPO
    "model.dropout",
    "trainer.beta",
    # BAY-LIN
    "trainer.final_hessian_mode",
    "trainer.l2_reg",
]

EVAL_METRIC_KEYS = [
    # main metrics
    "prefs/ece",
    # "prefs/elce",
    "prefs/euce",
    "win_rate",
    "prefs/confident_true_rate",
    "prefs/confident_false_rate",
    "ranking/0.0",
    "ranking/0.01",
    "ranking/0.2",
    "ranking/1.0",
    # additional metrics
    "rewards/chosen_pred_mean",
    "rewards/chosen_upper_mean",
    "rewards/chosen_lower_mean",
    "rewards/chosen_uncertainty_mean",
    "rewards/rejected_pred_mean",
    "rewards/rejected_upper_mean",
    "rewards/rejected_lower_mean",
    "rewards/rejected_uncertainty_mean",
    "prefs/pred_mean",
    "prefs/upper_mean",
    "prefs/lower_mean",
    "prefs/uncertainty_mean",
    # calibration curves
    "_output/calibration_statistics/pred",
    # "_output/calibration_statistics/lower",
    "_output/calibration_statistics/upper",
]

### Load RewardBench weights

In [ ]:
# from rewarduq.utils.rewardbench import compute_rewardbench_weights

# REWARD_BENCH_WEIGHTS = compute_rewardbench_weights()
# np.save(PATH_DATA / "rewardbench_weights.npy", REWARD_BENCH_WEIGHTS)

REWARD_BENCH_WEIGHTS = np.load(PATH_DATA / "rewardbench_weights.npy")

### Load output (optional)

Use the command to download all relevant output files from the cluster for generating the plots.
```bash
rsync -av <host>:<remote-path-to-output>/ plots/icml_paper/data/output/ \
    --include='*/' \
    --include='rewards_*.npy' \
    --include='job-*.log' \
    --include='.hydra/*' \
    --exclude="*" \
    --prune-empty-dirs
```

### Load metrics

In [ ]:
# from rewarduq.evaluation import load_metrics

# DF_METRICS = load_metrics(
#     [path for path in (PATH_DATA / "output").glob("*/*") if not path.name.startswith(".")],
#     config_keys=EVAL_CONFIG_KEYS,
#     metric_keys=EVAL_METRIC_KEYS,
#     metric_weights=REWARD_BENCH_WEIGHTS,
#     steps="final",
# )
# DF_METRICS.to_parquet(PATH_DATA / "metrics.parquet", index=False)

DF_METRICS = pd.read_parquet(PATH_DATA / "metrics.parquet")

DF_METRICS

## Plots

In [ ]:
Plotter.configure(save_always=True)

In [ ]:
def plot_metric(plot_group, method_names, metric_names, rect=None):
    BASE_MODEL_FAMILY_STYLE_MAP = {
        "Qwen3": dict(linestyle="-", marker="o", markersize=3),
        "Skywork-Qwen3": dict(linestyle="--", marker="o", markersize=3),
    }

    # create plots
    fig, axes = Plotter.create(
        nrows=len(metric_names),
        ncols=len(DATASET_LABEL_MAP),
        squeeze=False,
        sharex=True,
        sharey="row",
        layout=dict(
            layout="tight",
            pad=0.25,
            rect=rect,
        ),
    )
    for i, metric_name in enumerate(metric_names):
        for j, dataset_name in enumerate(DATASET_LABEL_MAP):
            for method_name in method_names:
                for model_family, base_model_names in BASE_MODEL_FAMILY_MAP.items():
                    base_model_sizes = [BASE_MODEL_SIZE_MAP[name] for name in base_model_names]
                    # filter results
                    df_metrics_filtered = DF_METRICS[
                        (DF_METRICS["dataset.train.path"] == dataset_name)
                        & (DF_METRICS["model.base_model_name_or_path"].isin(base_model_names))
                        & METHOD_SELECTOR_MAP[method_name](DF_METRICS)
                    ].set_index("model.base_model_name_or_path")
                    # plot ranking scores
                    axes[i, j].plot(
                        base_model_sizes,
                        [df_metrics_filtered[metric_name].get(name, float("nan")) for name in base_model_names],
                        label=f"{METHOD_LABEL_MAP[method_name]} ({model_family})",
                        color=METHOD_COLOR_MAP[method_name],
                        **BASE_MODEL_FAMILY_STYLE_MAP[model_family],
                    )
            Plotter.set(
                axes[i, j],
                title=DATASET_LABEL_MAP[dataset_name] if i == 0 else "",
                xlabel="model size (B)" if i == len(metric_names) - 1 else "",
                ylabel=METRICS_LABEL_MAP.get(metric_name, metric_name) if j == 0 else "",
            )
            axes[i, j].grid(True, linestyle="--", alpha=0.5)

    # Legend 1: UQ methods (left side)
    method_handles = [
        plt.Line2D([0], [0], color=METHOD_COLOR_MAP[method_name], **next(iter(BASE_MODEL_FAMILY_STYLE_MAP.values())))
        for method_name in method_names
    ]
    method_labels = [METHOD_LABEL_MAP[method_name] for method_name in method_names]
    fig.legend(
        method_handles,
        method_labels,
        title=r"\textbf{UQ method}",
        ncol=2,
        loc="lower left",
        bbox_to_anchor=(0.15, 0.0),
        alignment="center",
    )

    # Legend 2: Base model (right side)
    family_handles = [
        plt.Line2D([0], [0], color="gray", **BASE_MODEL_FAMILY_STYLE_MAP[family])
        for family in BASE_MODEL_FAMILY_STYLE_MAP
    ]
    family_labels = list(BASE_MODEL_FAMILY_STYLE_MAP)
    fig.legend(
        family_handles,
        family_labels,
        title=r"\textbf{Base model}",
        ncol=1,
        loc="lower right",
        bbox_to_anchor=(0.85, 0.0),
        alignment="center",
    )

    metric_names_str = "-".join(
        METRICS_FILENAME_MAP.get(name, name.replace("/", "_").replace(".", "_")) for name in metric_names
    )
    plot_group.add_plot(fig, f"metrics/{metric_names_str}")


with Plotter.group(
    figwidth=1,
    figheight_offset=0.03 / 5.5,  # hardcoded by trial and error
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        method_names=["ens_mlp", "ens_lora", "mcd_dpo", "bay_lin"],
        metric_names=["ranking/0.2"],
        rect=[0, 0.285, 1, 1],  # hardcoded by trial and error
    )

with Plotter.group(
    figwidth=1,
    figheight_offset=[0.14 / 5.5, 0.14 / 5.5, 0.10 / 5.5],  # hardcoded by trial and error
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        method_names=["ens_mlp", "ens_lora", "mcd_dpo", "bay_lin"],
        metric_names=[
            "win_rate",
            "prefs/confident_true_rate",
            "prefs/confident_false_rate",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        method_names=["ens_mlp", "ens_lora", "mcd_dpo", "bay_lin"],
        metric_names=[
            "ranking/0.0",
            "ranking/0.2",
            "ranking/1.0",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        method_names=["ens_mlp", "ens_lora", "mcd_dpo", "bay_lin"],
        metric_names=[
            "prefs/ece",
            "prefs/euce",
        ],
        rect=[0, 0.19, 1, 1],  # hardcoded by trial and error
    )

In [ ]:
def plot_calibration_curves(plot_group, method_names):
    def plot_calibration_curve(ax, curve, color="tab:blue", text=None):
        bin_prob_true, bin_prob_pred, bins, bin_count = curve
        n_bins = len(bins) - 1

        bin_count_rel = bin_count / np.max(bin_count)  # normalize by max
        # bin_count_rel = np.log(1 + bin_count_rel) / np.log(2)  # apply log scale
        bin_count_rel = 0.1 + 0.9 * np.nan_to_num(bin_count_rel)  # rescale to range [0.1, 1.0]
        bin_colors = [to_rgba(color, alpha) for alpha in bin_count_rel]
        ax.plot([0, 1], [0, 1], linestyle="--", color="tab:gray", label="perfect calibration")
        ax.bar(bins[:-1], bin_prob_true, 1 / n_bins, align="edge", color=bin_colors, label="actual calibration")

        if text is not None:
            ax.text(0.05, 0.95, text, fontsize=FONTSIZE_TINY, va="top", transform=ax.transAxes)

    # create plots
    base_model_names = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-4B"]

    for dataset_name in DATASET_LABEL_MAP:
        fig1, axes1 = Plotter.create(ncols=len(method_names), nrows=len(base_model_names), sharex=True, sharey=True)
        fig2, axes2 = Plotter.create(ncols=len(method_names), nrows=len(base_model_names), sharex=True, sharey=True)
        for i, base_model_name in enumerate(base_model_names):
            # filter results
            df_metrics_filtered = DF_METRICS[
                (DF_METRICS["dataset.train.path"] == dataset_name)
                & (DF_METRICS["model.base_model_name_or_path"] == base_model_name)
            ]

            # plot calibration curves for predictions
            for j, method_name in enumerate(method_names):
                method_selector = METHOD_SELECTOR_MAP[method_name](df_metrics_filtered)
                if method_selector.sum() == 1:
                    ece = df_metrics_filtered.loc[method_selector, "prefs/ece"].item()
                    euce = df_metrics_filtered.loc[method_selector, "prefs/euce"].item()
                    curve_pred = df_metrics_filtered.loc[method_selector, "_output/calibration_statistics/pred"].item()
                    curve_upper = df_metrics_filtered.loc[
                        method_selector, "_output/calibration_statistics/upper"
                    ].item()
                    plot_calibration_curve(
                        axes1[i, j], curve_pred, text=f"ECE: {ece:.2f}", color=METHOD_COLOR_MAP[method_name]
                    )
                    plot_calibration_curve(
                        axes2[i, j], curve_upper, text=f"EBCE: {euce:.2f}", color=METHOD_COLOR_MAP[method_name]
                    )
                    Plotter.set(
                        axes1[i, j],
                        title=METHOD_LABEL_MAP[method_name] if i == 0 else "",
                        xticks=[0.0, 0.5, 1.0],
                        yticks=[0.0, 0.5, 1.0],
                    )
                    Plotter.set(
                        axes2[i, j],
                        title=METHOD_LABEL_MAP[method_name] if i == 0 else "",
                        xticks=[0.0, 0.5, 1.0],
                        yticks=[0.0, 0.5, 1.0],
                    )
                else:
                    logger.warning(
                        f"Found {method_selector.sum()} entries for '{method_name}' in dataset '{dataset_name}' and"
                        f" base model '{base_model_name}'. Skipping calibration curve plot for this method."
                    )
        fig1.supxlabel("predicted probability $p_\\theta(B_m)$", x=0.525)
        fig2.supxlabel("predicted upper bound $\\overline{p_\\theta}(B_m)$", x=0.525)
        fig1.supylabel("true prob. $\\mathbb{P}(B_m)$", y=0.525)
        fig2.supylabel("true prob. $\\mathbb{P}(B_m)$", y=0.525)
        base_model_names_str = "-".join(
            BASE_MODEL_FILENAME_MAP[base_model_name] for base_model_name in base_model_names
        )
        plot_group.add_plot(
            fig1,
            f"calibration_curve/{DATASET_FILENAME_MAP[dataset_name]}-{base_model_names_str}-ece",
        )
        plot_group.add_plot(
            fig2,
            f"calibration_curve/{DATASET_FILENAME_MAP[dataset_name]}-{base_model_names_str}-ebce",
        )


with Plotter.group(
    figwidth=0.65,
    axratio=1,
    grid_ncols=2,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_calibration_curves(
        plot_group,
        method_names=["ens_mlp", "ens_lora", "mcd_dpo", "bay_lin"],
    )

In [ ]:
def plot_ranking_ranges(plot_group, alphas, **kwargs):
    def ranking(win_rate, ct_rate, cf_rate, alpha):
        ranking_pos = ct_rate / (win_rate + alpha * (1 - win_rate))
        ranking_neg = cf_rate / (1 - win_rate + alpha * (win_rate))
        return ranking_pos, ranking_neg

    def plot_ranking_range(ax, alpha, color):
        win_rates = np.linspace(0, 1, 100)

        ranking_pos_ub, ranking_neg_ub = ranking(win_rates, win_rates, 0, alpha=alpha)
        ranking_scores_ub = ranking_pos_ub - ranking_neg_ub
        ranking_pos_lb, ranking_neg_lb = ranking(win_rates, 0, 1 - win_rates, alpha=alpha)
        ranking_scores_lb = ranking_pos_lb - ranking_neg_lb

        ax.fill_between(win_rates, ranking_scores_lb, ranking_scores_ub, color=color, alpha=0.3)
        ax.plot(win_rates, ranking_scores_ub, linestyle="--", color=color, label="upper bound")
        ax.plot(win_rates, ranking_scores_lb, linestyle="--", color=color, label="lower bound")

    # create plots
    fig, axes = Plotter.create(ncols=len(alphas), sharey=True)
    for i, (alpha, color) in enumerate(zip(alphas, plt.get_cmap("tab10").colors[5:])):
        plot_ranking_range(axes[i], alpha, color)
        if kwargs["with_title"]:
            axes[i].set_title(f"$\\alpha={alpha}$")
        if i == 0:
            axes[i].set_ylabel("$\\mathrm{RS}_{\\alpha}$")
            axes[i].yaxis.set_label_coords(kwargs["ylabel_coords_x"], 0.5)
        Plotter.set(
            axes[i],
            xticks=[0.0, 0.5, 1.0],
            yticks=[-1.0, -0.5, 0.0, 0.5, 1.0],
        )
    fig.supxlabel("$\\mathrm{win\\ rate}$", x=kwargs["xlabel_x"])
    plot_group.add_plot(fig, "ranking/range")


def plot_ranking_factors(plot_group, alphas, **kwargs):
    def ranking_factor(x, alpha):
        return x / (x + alpha * (1 - x))

    # create plots
    fig, ax = Plotter.create()
    x = np.linspace(0, 1, 100)
    o = np.array(kwargs["annotate_x"])
    for alpha, color in zip(alphas, plt.get_cmap("tab10").colors[5:]):
        ax.plot(x, ranking_factor(x, alpha), color=color, label=f"$\\alpha = {alpha}$")
        if kwargs["annotate"] == "all" or alpha in kwargs["annotate"]:
            ax.plot(o, ranking_factor(o, alpha), color=color, linestyle="none", marker="o", markersize=2)
            for xi, yi in zip(o, ranking_factor(o, alpha)):
                plt.text(
                    xi + kwargs["annotate_offset"],
                    yi - kwargs["annotate_offset"],
                    f"{yi:.2f}",
                    va="top",
                    color=color,
                    fontsize=FONTSIZE_SMALL,
                )
    Plotter.set(
        ax,
        xticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        xlabel="$x$",
        ylabel="$f_{\\alpha}(x)$",
        legend=dict(fontsize=FONTSIZE_SMALL, handlelength=1.0, handletextpad=0.5, labelspacing=0.1),
    )
    plot_group.add_plot(fig, "ranking/factors")


with Plotter.group(
    figwidth=[0.8, 0.4],
    axratio=[2, 1],
    # grid_ncols=2,
    # consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_ranking_ranges(
        plot_group,
        alphas=[0.0, 0.1, 0.2, 0.5, 1.0],
        # additional kwargs
        with_title=True,
        ylabel_coords_x=-0.45,
        xlabel_x=0.525,
    )

    plot_ranking_factors(
        plot_group,
        alphas=[0.0, 0.1, 0.2, 0.5, 1.0],
        # additional kwargs
        annotate=[0.2],
        annotate_x=[0.2, 0.4, 0.6, 0.8],
        annotate_offset=0.015,
    )

In [ ]:
Plotter.configure(save_always=False)
plt.close()

## Tables

In [ ]:
def print_hyperparameters(key_map):
    def float_format(x):
        """Format float without trailing zeros."""
        formatted = f"{x:.10f}".rstrip("0").rstrip(".")
        return formatted

    def index_format(x):
        """Format index by removing prefix before '/'."""
        if "/" in x:
            x = x.split("/")[-1]
        return x

    df_list = []
    for method_name, parameter_names in key_map.items():
        df_config = DF_METRICS[
            (DF_METRICS["dataset.train.path"] == "trl-lib/ultrafeedback_binarized")
            & METHOD_SELECTOR_MAP[method_name](DF_METRICS)
        ]
        df_config = df_config.set_index("model.base_model_name_or_path")
        df_config.index = df_config.index.map(index_format)
        df_config = df_config[list(parameter_names.keys())].rename(columns=parameter_names)
        df_config.columns = pd.MultiIndex.from_product([[METHOD_LABEL_MAP[method_name]], df_config.columns])
        df_list.append(df_config)

    df_final = pd.concat(df_list, axis=1)
    display(df_final)
    print(df_final.to_latex(float_format=float_format, na_rep=""))


print_hyperparameters(
    {
        "ens_lin": {
            "trainer.learning_rate": r"\eta",
            "trainer.regularization_towards_initial_weights": r"\lambda",
            "trainer.center_rewards_coefficient": r"\gamma",
        },
        "ens_mlp": {
            "trainer.learning_rate": r"\eta",
            "trainer.regularization_towards_initial_weights": r"\lambda",
            "trainer.center_rewards_coefficient": r"\gamma",
        },
        "ens_lora": {
            "trainer.learning_rate": r"\eta",
            "trainer.regularization_towards_initial_weights": r"\lambda",
            "trainer.center_rewards_coefficient": r"\gamma",
        },
        "mcd_dpo": {
            "trainer.learning_rate": r"\eta",
            "trainer.beta": r"\lambda",
            "model.dropout": r"p_dropout",
        },
        "bay_lin": {
            "trainer.learning_rate": r"\eta",
            "trainer.l2_reg": r"\lambda",
        },
    }
)

## Ablations

### ENS-LIN

In [ ]:
with Plotter.group(
    figwidth=1,
    figheight_offset=0.03 / 5.5,  # hardcoded by trial and error
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        DF_METRICS,
        method_names=["ens_mlp", "ens_lin"],
        metric_names=["ranking/0.2"],
        rect=[0, 0.285, 1, 1],  # hardcoded by trial and error
    )
    plot_group.rename(lambda name: f"ablations/ens_lin/{name}")

with Plotter.group(
    figwidth=1,
    figheight_offset=[0.14 / 5.5, 0.14 / 5.5, 0.10 / 5.5],  # hardcoded by trial and error
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        DF_METRICS,
        method_names=["ens_mlp", "ens_lin"],
        metric_names=[
            "win_rate",
            "prefs/confident_true_rate",
            "prefs/confident_false_rate",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        DF_METRICS,
        method_names=["ens_mlp", "ens_lin"],
        metric_names=[
            "ranking/0.0",
            "ranking/0.2",
            "ranking/1.0",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        DF_METRICS,
        method_names=["ens_mlp", "ens_lin"],
        metric_names=[
            "prefs/ece",
            "prefs/euce",
        ],
        rect=[0, 0.19, 1, 1],  # hardcoded by trial and error
    )
    plot_group.rename(lambda name: f"ablations/ens_lin/{name}")

In [ ]:
with Plotter.group(
    figwidth=0.5,
    axratio=1,
    grid_ncols=2,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_calibration_curves(
        plot_group,
        DF_METRICS,
        method_names=["ens_mlp", "ens_lin"],
    )
    plot_group.rename(lambda name: f"ablations/ens_lin/{name}")

### BAY-LIN weighted

In [ ]:
with Plotter.group(
    figwidth=1,
    figheight_offset=0.03 / 5.5,  # hardcoded by trial and error
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        method_names=["bay_lin", "bay_lin_weighted"],
        metric_names=["ranking/0.2"],
        rect=[0, 0.285, 1, 1],  # hardcoded by trial and error
    )
    plot_group.rename(lambda name: f"ablations/bay_lin_weighted/{name}")

with Plotter.group(
    figwidth=1,
    figheight_offset=[0.14 / 5.5, 0.14 / 5.5, 0.10 / 5.5],  # hardcoded by trial and error
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_metric(
        plot_group,
        method_names=["bay_lin", "bay_lin_weighted"],
        metric_names=[
            "win_rate",
            "prefs/confident_true_rate",
            "prefs/confident_false_rate",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        method_names=["bay_lin", "bay_lin_weighted"],
        metric_names=[
            "ranking/0.0",
            "ranking/0.2",
            "ranking/1.0",
        ],
        rect=[0, 0.135, 1, 1],  # hardcoded by trial and error
    )
    plot_metric(
        plot_group,
        method_names=["bay_lin", "bay_lin_weighted"],
        metric_names=[
            "prefs/ece",
            "prefs/euce",
        ],
        rect=[0, 0.19, 1, 1],  # hardcoded by trial and error
    )
    plot_group.rename(lambda name: f"ablations/bay_lin_weighted/{name}")

In [ ]:
with Plotter.group(
    figwidth=0.5,
    axratio=1,
    grid_ncols=2,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_calibration_curves(
        plot_group,
        method_names=["bay_lin", "bay_lin_weighted"],
    )
    plot_group.rename(lambda name: f"ablations/bay_lin_weighted/{name}")

## temp

In [ ]:
from rewarduq.evaluation import load_metrics

DF_METRICS_ALL = load_metrics(
    [path for path in (PATH_DATA / "output").glob("*/*") if not path.name.startswith(".")],
    config_keys=EVAL_CONFIG_KEYS,
    metric_keys=EVAL_METRIC_KEYS,
    metric_weights=REWARD_BENCH_WEIGHTS,
    steps="all",
)
DF_METRICS_ALL

In [ ]:
from rewarduq.evaluation import load_predictions

DF_PREDICTIONS = load_predictions(
    [path for path in (PATH_DATA / "output").glob("*/*") if not path.name.startswith(".")],
    config_keys=EVAL_CONFIG_KEYS,
    beta_mapping={
        "mlp_head_ensemble.MLPHeadEnsemblePipeline": "model.bounds_function_std_beta",
        "dpo_head_dropout_ensemble.DPOHeadDropoutEnsemblePipeline": "model.scale_bounds",
        "bayesian_linear_head.BayesianLinearHeadPipeline": "model.std_beta",
        "lora_ensemble.LoraEnsemblePipeline": "model.bounds_function_std_beta",
    },
    steps="final",
)
DF_PREDICTIONS

In [ ]:
def plot_preferences(plot_group):
    # create plots
    for dataset_name in DATASET_LABEL_MAP:
        for base_model_name in ["Qwen/Qwen3-4B", "Skywork/Skywork-Reward-V2-Qwen3-4B"]:
            # filter results
            df_metrics_filtered = DF_METRICS_ALL[
                (DF_METRICS_ALL["dataset.train.path"] == dataset_name)
                & (DF_METRICS_ALL["model.base_model_name_or_path"] == base_model_name)
            ]

            # plot preferences
            fig, axes = Plotter.create(ncols=len(METHOD_LABEL_MAP), sharey=True)
            for i, method_name in enumerate(METHOD_LABEL_MAP):
                method_selector = METHOD_SELECTOR_MAP[method_name](df_metrics_filtered)
                steps = df_metrics_filtered.loc[method_selector, "step"]
                axes[i].plot(
                    steps,
                    df_metrics_filtered.loc[method_selector, "prefs/pred_mean"],
                )
                axes[i].fill_between(
                    steps,
                    df_metrics_filtered.loc[method_selector, "prefs/lower_mean"],
                    df_metrics_filtered.loc[method_selector, "prefs/upper_mean"],
                    alpha=0.25,
                )
                Plotter.set(axes[i], title=METHOD_LABEL_MAP[method_name])
            fig.supxlabel("steps", x=0.525)
            axes[0].set_ylabel("preference probability")
            plot_group.add_plot(
                fig,
                f"preferences/{DATASET_FILENAME_MAP[dataset_name]}-{BASE_MODEL_FILENAME_MAP[base_model_name]}",
            )


with Plotter.group(
    figwidth=1.0,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_preferences(plot_group)

In [ ]:
def plot_uncertainty_distribution(plot_group):
    # create plots
    for dataset_name in DATASET_LABEL_MAP:
        for base_model_name in ["Qwen/Qwen3-4B", "Skywork/Skywork-Reward-V2-Qwen3-4B"]:
            # filter results
            df_predictions_filtered = DF_PREDICTIONS[
                (DF_PREDICTIONS["dataset.train.path"] == dataset_name)
                & (DF_PREDICTIONS["model.base_model_name_or_path"] == base_model_name)
            ]

            # plot preferences
            fig, axes = Plotter.create(ncols=len(METHOD_LABEL_MAP), sharex=True, sharey=True)
            for i, method_name in enumerate(METHOD_LABEL_MAP):
                method_selector = METHOD_SELECTOR_MAP[method_name](df_predictions_filtered)
                axes[i].hist(
                    df_predictions_filtered.loc[method_selector, "rewards_std"],
                    bins=20,
                    # density=True,
                )
                Plotter.set(axes[i], title=METHOD_LABEL_MAP[method_name])
            fig.supxlabel("reward uncertainty", x=0.525)
            axes[0].set_ylabel("frequency")
            plot_group.add_plot(
                fig,
                f"uncertainty_distribution/{DATASET_FILENAME_MAP[dataset_name]}-{BASE_MODEL_FILENAME_MAP[base_model_name]}",
            )


with Plotter.group(
    figwidth=1.5,
    grid_ncols=1,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_uncertainty_distribution(plot_group)